# Phase 3 — Grade completions (local, no Colab)

See `docs/research_proposal.md` §4.4. Runs entirely locally -- no GPU/model needed, just the two JSON files from Phase 2 (downloaded from Drive into `artifacts/` here) plus `ANTHROPIC_API_KEY`. Produces `labels_{text,mm}_{A,B}.jsonl`, which need to go **back** to Drive's `artifacts/` before running Phase 4 (`03_extract_directions.ipynb`) in Colab.

In [1]:
%pip install -q anthropic pillow python-dotenv tqdm


Note: you may need to restart the kernel to use updated packages.


## Prerequisites checklist

- `artifacts/phase2_completions_A.json` and `phase2_completions_B.json` downloaded from Drive into this project's local `artifacts/` folder (gitignored, not created yet).
- `.env` at the project root with `ANTHROPIC_API_KEY=...` filled in (gitignored -- never commit this file).
- `data/eval/images/*.png` and `reference/gulati_raval_2602.16931/judge_prompt.txt` are already present locally -- nothing to move for those.

In [2]:
import os

from dotenv import load_dotenv

os.chdir('/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project')
load_dotenv(dotenv_path='.env')  # explicit path -- find_dotenv()'s auto-detection can fail depending on execution context
print('ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))


ANTHROPIC_API_KEY set: True


## Grade both organisms

In [3]:
import json
from pathlib import Path

from src.generate import build_labeling_examples, load_multimodal_eval_set
from src.judge import grade_batch_api

ARTIFACTS = Path('artifacts')
mm_images = [ex['image'] for ex in load_multimodal_eval_set()]

for organism in ['A', 'B']:
    data = json.loads((ARTIFACTS / f'phase2_completions_{organism}.json').read_text())

    text_examples = build_labeling_examples(
        data['text_prompts'], data['text_base_completions'], data['text_ft_completions'], id_prefix='text_',
    )
    grade_batch_api(text_examples, out_path=ARTIFACTS / f'labels_text_{organism}.jsonl')

    mm_examples = build_labeling_examples(
        data['mm_prompts'], data['mm_base_completions'], data['mm_ft_completions'], images=mm_images, id_prefix='mm_',
    )
    grade_batch_api(mm_examples, out_path=ARTIFACTS / f'labels_mm_{organism}.jsonl')

    print(f'organism {organism}: grading done')


/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
grading labels_mm_A.jsonl: 100%|██████████| 30/30 [00:00<?, ?it/s]


organism A: grading done


grading labels_mm_B.jsonl: 100%|██████████| 30/30 [00:56<00:00,  9.45s/it]

organism B: grading done


## Next step

Upload the four `labels_*.jsonl` files this produced (in local `artifacts/`) to Drive's `artifacts/` folder before running `03_extract_directions.ipynb` in Colab.